# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the [FAIR^2](https://doi.org/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library. 

### Dataset Source
The dataset source is provided via a Croissant schema URL.

- **Title:** Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells
- **Citation:** Kulka, M., Rodriguez Miera, S., Marcet-Palacios, M. 2026. Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells. Frontiers.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint
import warnings
warnings.filterwarnings('ignore')

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Show full metadata name and description
print(f"Dataset Name: {dataset.metadata.name}\n\nDataset Description: {dataset.metadata.description}\n")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

This step explores the high-level dataset structure and helps select `recordSet` and fields for further extraction.

In [ ]:
# List all record sets in the dataset by @id and show their columns
rs_overview = []
for record_set in dataset.metadata.record_sets:
    fields = [field['@id'] for field in record_set.fields] if hasattr(record_set, 'fields') else []
    cols = [col['@id'] for col in record_set.columns] if hasattr(record_set, 'columns') else []
    rs_overview.append({'name': getattr(record_set, 'name', None),
                       '@id': record_set['@id'],
                       'description': getattr(record_set, 'description', None),
                       'fields': fields,
                       'columns': cols})

if rs_overview:
    print("Found Record Sets:")
    for idx, rs in enumerate(rs_overview):
        print(f"[{idx}] Name: {rs['name']} | @id: {rs['@id']}")
        print(f"    Description: {rs['description']}")
        print(f"    Fields/Columns: {rs['fields'] if len(rs['fields']) > 0 else rs['columns']}")
        print()
else:
    print("No record sets found in this dataset's schema.")

## 3. Data Extraction
Load data from specific record sets into DataFrames. Below we extract from all declared record sets and display columns from the primary one.

In [ ]:
# Extract all records from each record set into pandas DataFrames
record_sets = [rs['@id'] for rs in rs_overview] if rs_overview else []
dataframes = {}

if record_sets:
    for record_set_id in record_sets:
        # Each record is a dict mapping field @id to value
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records from RecordSet {record_set_id}")
    # Example with the first record set
    example_rs_id = record_sets[0]
    print(f"\nColumns in DataFrame for RecordSet {example_rs_id}:")
    print(dataframes[example_rs_id].columns.tolist())
    dataframes[example_rs_id].head()
else:
    print('No record sets available for data extraction.')

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping data. All fields are referenced using their `@id`.

In [ ]:
# For illustration, select a numeric field and a group field from the first record set
if record_sets:
    rs_id = record_sets[0]
    df = dataframes[rs_id]
    # Try to auto-detect a numeric field and categorical/group field
    sample_numeric_field = None
    group_field = None

    # Use pandas to heuristically pick columns
    numeric_candidates = df.select_dtypes(include=['number']).columns.tolist()
    if len(numeric_candidates) > 0:
        sample_numeric_field = numeric_candidates[0]
    else:
        # Try to convert columns to numeric as secondary attempt
        for col in df.columns:
            try:
                _ = pd.to_numeric(df[col].dropna())
                sample_numeric_field = col
                break
            except Exception:
                continue
    # Categoricals: pick first non-numeric as group
    non_numeric = [c for c in df.columns if c != sample_numeric_field]
    if non_numeric:
        group_field = non_numeric[0]

    print(f"Using numeric field: {sample_numeric_field} and group field: {group_field}\n")
    # Force convert numeric if not already
    df[sample_numeric_field] = pd.to_numeric(df[sample_numeric_field], errors='coerce')
    # Example: filter where numeric_field > threshold
    threshold = df[sample_numeric_field].mean() if pd.notnull(df[sample_numeric_field]).any() else 0
    filtered_df = df[df[sample_numeric_field] > threshold]
    print(f"Filtered records with {sample_numeric_field} > {threshold:.2f}:")
    print(filtered_df.head())

    # Normalize
    col_norm = f"{sample_numeric_field}_normalized"
    filtered_df[col_norm] = (filtered_df[sample_numeric_field] - filtered_df[sample_numeric_field].mean()) / filtered_df[sample_numeric_field].std()
    print(f"\nNormalized {sample_numeric_field} for filtered records:")
    print(filtered_df[[sample_numeric_field, col_norm]].head())

    # Group by group_field (categorical)
    if group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field)[sample_numeric_field].mean().reset_index()
        print(f"\nGrouped mean of {sample_numeric_field} by {group_field}:")
        print(grouped_df.head())
else:
    print('No available data for EDA.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if record_sets:
    rs_id = record_sets[0]
    df = dataframes[rs_id]
    if sample_numeric_field is not None:
        # Histogram
        plt.figure(figsize=(8, 4))
        df[sample_numeric_field].dropna().hist(bins=40)
        plt.title(f'Distribution of {sample_numeric_field}')
        plt.xlabel(sample_numeric_field)
        plt.ylabel('Count')
        plt.show()
        # If group field, also boxplot
        if group_field in df.columns and df[group_field].nunique() < 30:
            plt.figure(figsize=(10, 4))
            sns.boxplot(data=df, x=group_field, y=sample_numeric_field)
            plt.title(f'{sample_numeric_field} by {group_field}')
            plt.xticks(rotation=90)
            plt.show()
else:
    print('No data available for visualization.')

## 6. Conclusion
This notebook demonstrated how to use the `mlcroissant` library to load and explore the FAIR^2 dataset defined using a Croissant schema. By referencing all data entities by their `@id`, we loaded available record sets, performed basic filtering and normalization operations, and visualized key numeric distributions. These steps establish a pattern for robust, schema-driven data handling in computational biology and proteomics research.
